In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta, date

# ==========================================
# 1. SYNTHETIC DATA GENERATION
# ==========================================
def generate_financial_data(num_rows=12000):
    print("Generating synthetic financial data...")
    
    # Base parameters
    start_date = date(2022, 1, 1)
    departments = ['Sales', 'Marketing', 'Engineering', 'Customer Support', 'Operations']
    regions = ['North America', 'EMEA', 'APAC', 'LATAM']
    statuses = ['Completed', 'Completed', 'Completed', 'Pending', 'Failed'] # Weighted
    
    # Generate random data
    dates = [start_date + timedelta(days=np.random.randint(0, 730)) for _ in range(num_rows)]
    
    df = pd.DataFrame({
        'Transaction_ID': [f"TXN-{i:06d}" for i in range(1, num_rows + 1)],
        'Date': dates,
        'Department': np.random.choice(departments, num_rows),
        'Region': np.random.choice(regions, num_rows),
        'Transaction_Status': np.random.choice(statuses, num_rows),
        'Revenue_Generated': np.random.normal(5000, 1500, num_rows).round(2),
        'Operational_Cost': np.random.normal(2000, 800, num_rows).round(2)
    })
    
    # Introduce Nulls to simulate messy real-world data
    df.loc[df.sample(frac=0.02).index, 'Revenue_Generated'] = np.nan
    df.loc[df.sample(frac=0.01).index, 'Operational_Cost'] = np.nan
    
    # Introduce Strategic Anomalies
    # Anomaly 1: Massive Cost Spike in APAC operations (Supply chain issue in mid-2023)
    anomaly_mask_1 = (df['Region'] == 'APAC') & (df['Date'] > date(2023, 5, 1)) & (df['Date'] < date(2023, 6, 15))
    df.loc[anomaly_mask_1, 'Operational_Cost'] = df.loc[anomaly_mask_1, 'Operational_Cost'] * np.random.uniform(2.5, 4.0)
    
    # Anomaly 2: Sudden Revenue Dip in North America Marketing (Algorithm penalty)
    anomaly_mask_2 = (df['Department'] == 'Marketing') & (df['Region'] == 'North America') & (df['Date'] > date(2023, 10, 1)) & (df['Date'] < date(2023, 11, 1))
    df.loc[anomaly_mask_2, 'Revenue_Generated'] = df.loc[anomaly_mask_2, 'Revenue_Generated'] * 0.3

    # Ensure no negative costs/revenues from normal distribution
    df['Revenue_Generated'] = df['Revenue_Generated'].clip(lower=0)
    df['Operational_Cost'] = df['Operational_Cost'].clip(lower=0)

    # Save raw data
    df.to_csv('raw_financial_data.csv', index=False)
    print("Raw data generated and saved to 'raw_financial_data.csv'")
    return df

# ==========================================
# 2. AUTOMATED CLEANING & FEATURE ENGINEERING
# ==========================================
def clean_and_transform_data(df):
    print("Initiating data cleaning and automation workflow...")
    
    # 1. Handle Missing Values
    # Forward fill dates if any were missing (not applicable here, but good practice)
    # Fill missing financials with median of their respective Department & Region
    df['Revenue_Generated'] = df.groupby(['Department', 'Region'])['Revenue_Generated'].transform(lambda x: x.fillna(x.median()))
    df['Operational_Cost'] = df.groupby(['Department', 'Region'])['Operational_Cost'].transform(lambda x: x.fillna(x.median()))
    
    # 2. Filter out 'Failed' transactions for financial reporting
    df_clean = df[df['Transaction_Status'] != 'Failed'].copy()
    
    # 3. Calculate New Metrics: Net_Profit & Net_Profit_Margin
    df_clean['Net_Profit'] = df_clean['Revenue_Generated'] - df_clean['Operational_Cost']
    
    # Avoid division by zero
    df_clean['Net_Profit_Margin'] = np.where(
        df_clean['Revenue_Generated'] > 0,
        (df_clean['Net_Profit'] / df_clean['Revenue_Generated']).round(4),
        0
    )
    
    # 4. Sort and reset index
    df_clean = df_clean.sort_values('Date').reset_index(drop=True)
    
    # Save clean data for Tableau/SQL
    df_clean.to_csv('cleaned_financial_data.csv', index=False)
    print("Data cleaned and saved to 'cleaned_financial_data.csv'")
    
    return df_clean

# Execute the workflow
if __name__ == "__main__":
    raw_df = generate_financial_data()
    clean_df = clean_and_transform_data(raw_df)
    print(clean_df.head())

Generating synthetic financial data...
Raw data generated and saved to 'raw_financial_data.csv'
Initiating data cleaning and automation workflow...
Data cleaned and saved to 'cleaned_financial_data.csv'
  Transaction_ID        Date        Department         Region  \
0     TXN-007123  2022-01-01  Customer Support  North America   
1     TXN-008649  2022-01-01       Engineering           EMEA   
2     TXN-000995  2022-01-01       Engineering           APAC   
3     TXN-003225  2022-01-01  Customer Support           APAC   
4     TXN-006769  2022-01-01       Engineering           EMEA   

  Transaction_Status  Revenue_Generated  Operational_Cost  Net_Profit  \
0          Completed            4829.00          1488.850    3340.150   
1            Pending            5105.67          1939.245    3166.425   
2          Completed            2324.78          2056.770     268.010   
3          Completed            5403.39          1660.180    3743.210   
4          Completed            3169.84  